In [ ]:
import requests
import pandas as pd

BASE = "https://api.opendota.com/api"

resp = requests.get(f"{BASE}/proMatches")
resp.raise_for_status()
matches = resp.json()

df_sample = pd.DataFrame(matches)
print(df_sample.shape)
df_sample.head()

In [ ]:
sample_match_id = df_sample.iloc[0]['match_id']

resp2 = requests.get(f"{BASE}/matches/{sample_match_id}")
resp2.raise_for_status()
detail = resp2.json()

print("Top-level keys:", list(detail.keys()))
print("\nversion field (None = unparsed, a number = parsed):", detail.get("version"))
print("\nnumber of player records:", len(detail.get("players", [])))

In [ ]:
import time
import os

os.makedirs("data", exist_ok=True)

def get_pro_matches_page(less_than_match_id=None, max_retries=3):
    params = {"less_than_match_id": less_than_match_id} if less_than_match_id else {}
    for attempt in range(max_retries):
        resp = requests.get(f"{BASE}/proMatches", params=params)
        if resp.status_code == 429:
            wait = 5 * (attempt + 1)
            print(f"  rate limited, waiting {wait}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()
    raise RuntimeError(f"gave up on page after {max_retries} retries (cursor={less_than_match_id})")

def get_pro_matches(n_pages=5):
    all_matches = []
    cursor = None
    for i in range(n_pages):
        batch = get_pro_matches_page(cursor)
        if not batch:
            break
        all_matches.extend(batch)
        cursor = batch[-1]["match_id"]
        print(f"page {i+1}/{n_pages}: {len(batch)} matches, cursor={cursor}")
        time.sleep(1.1)
    return all_matches

pro_matches = get_pro_matches(n_pages=5)  # ~500 matches
df_matches = pd.DataFrame(pro_matches)
print(df_matches.shape)
df_matches.head()

In [ ]:
df_matches.isnull().sum()

In [ ]:
before = len(df_matches)

df_matches_clean = df_matches.dropna(subset=['radiant_team_id', 'dire_team_id']).copy()

after = len(df_matches_clean)
print(f"before: {before}, after: {after}, dropped: {before - after} ({(before-after)/before:.1%})")

# sanity checks on what remains
df_matches_clean['start_time'] = pd.to_datetime(df_matches_clean['start_time'], unit='s')
print(f"\ndate range: {df_matches_clean['start_time'].min()} to {df_matches_clean['start_time'].max()}")
print(f"unique radiant teams: {df_matches_clean['radiant_team_id'].nunique()}")
print(f"unique dire teams: {df_matches_clean['dire_team_id'].nunique()}")
df_matches_clean[['match_id', 'start_time', 'radiant_name', 'dire_name', 'radiant_win']].head(10)

In [ ]:
league_counts = df_matches_clean['league_name'].value_counts()
print(league_counts.head(20))
print(f"\ntotal unique leagues: {df_matches_clean['league_name'].nunique()}")

In [ ]:
resp = requests.get(f"{BASE}/leagues")
resp.raise_for_status()
all_leagues = pd.DataFrame(resp.json())

our_leagueids = df_matches_clean['leagueid'].unique()
leagues_used = all_leagues[all_leagues['leagueid'].isin(our_leagueids)]

print(leagues_used[['leagueid', 'name', 'tier']].sort_values('tier', ascending=False))

In [ ]:
betboom_matches = df_matches_clean[df_matches_clean['league_name'] == 'BETBOOM Streamers Battle Dota 15']
print(betboom_matches[['radiant_name', 'dire_name']].drop_duplicates())

In [ ]:
EXCLUDE_LEAGUE_IDS = [20134, 20159, 20206, 20176]  # excluded-tier + BETBOOM Streamers Battle

df_final = df_matches_clean[~df_matches_clean['leagueid'].isin(EXCLUDE_LEAGUE_IDS)].copy()
print(f"before: {len(df_matches_clean)}, after: {len(df_final)}")
df_final['league_name'].value_counts()

In [ ]:
from datetime import datetime, timedelta, timezone

WINDOW_DAYS = 90  # rolling window size

def get_pro_matches_windowed(window_days=WINDOW_DAYS):
    cutoff = datetime.now(timezone.utc) - timedelta(days=window_days)
    all_matches = []
    cursor = None
    page = 0

    while True:
        page += 1
        batch = get_pro_matches_page(cursor)
        if not batch:
            break

        batch_df = pd.DataFrame(batch)
        batch_df['start_time'] = pd.to_datetime(batch_df['start_time'], unit='s', utc=True)

        in_window = batch_df[batch_df['start_time'] >= cutoff]
        all_matches.append(in_window)

        oldest_in_batch = batch_df['start_time'].min()
        print(f"page {page}: {len(batch)} matches, oldest={oldest_in_batch.date()}")

        if oldest_in_batch < cutoff:
            break

        cursor = batch_df['match_id'].iloc[-1]
        time.sleep(1.1)

    return pd.concat(all_matches, ignore_index=True)

df_matches = get_pro_matches_windowed(window_days=WINDOW_DAYS)
print(f"\ntotal matches in {WINDOW_DAYS}-day window: {len(df_matches)}")

In [ ]:
team_appearances = pd.concat([df_matches['radiant_team_id'], df_matches['dire_team_id']]).value_counts()
print(team_appearances.describe())
print(f"\nteams with <5 matches: {(team_appearances < 5).sum()} / {len(team_appearances)}")
print(f"teams with <10 matches: {(team_appearances < 10).sum()} / {len(team_appearances)}")

## Widen the window and compare


In [ ]:
df_matches_180 = get_pro_matches_windowed(window_days=180)

team_appearances_180 = pd.concat([
    df_matches_180['radiant_team_id'], df_matches_180['dire_team_id']
]).value_counts()

print(f"total matches in 180-day window: {len(df_matches_180)}")
print(team_appearances_180.describe())
print(f"\nteams with <5 matches: {(team_appearances_180 < 5).sum()} / {len(team_appearances_180)}")
print(f"teams with <10 matches: {(team_appearances_180 < 10).sum()} / {len(team_appearances_180)}")


## Durable league filtering (replaces the hardcoded `EXCLUDE_LEAGUE_IDS`)


In [ ]:
resp = requests.get(f"{BASE}/leagues")
resp.raise_for_status()
all_leagues = pd.DataFrame(resp.json())

professional_leagueids = set(all_leagues[all_leagues['tier'] == 'professional']['leagueid'])

MANUALLY_EXCLUDED_LEAGUEIDS = {20176}  # BETBOOM Streamers Battle Dota 15

df_tier_filtered = df_matches_180[
    df_matches_180['leagueid'].isin(professional_leagueids)
    & ~df_matches_180['leagueid'].isin(MANUALLY_EXCLUDED_LEAGUEIDS)
].copy()

print(f"before tier filter: {len(df_matches_180)}, after: {len(df_tier_filtered)}")
print("\nleagues remaining")
print(df_tier_filtered['league_name'].value_counts())


## Minimum-match threshold (explicit rule, not a silent filter)


In [ ]:
MIN_MATCHES = 5

team_appearances_final = pd.concat([
    df_tier_filtered['radiant_team_id'], df_tier_filtered['dire_team_id']
]).value_counts()

qualifying_teams = team_appearances_final[team_appearances_final >= MIN_MATCHES].index

df_reliable = df_tier_filtered[
    df_tier_filtered['radiant_team_id'].isin(qualifying_teams)
    & df_tier_filtered['dire_team_id'].isin(qualifying_teams)
].copy()

print(f"matches with both teams >= {MIN_MATCHES} appearances: {len(df_reliable)} / {len(df_tier_filtered)}")
print(f"qualifying teams: {len(qualifying_teams)} / {len(team_appearances_final)}")


In [ ]:
import json

def get_match_detail(match_id, max_retries=3):
    for attempt in range(max_retries):
        resp = requests.get(f"{BASE}/matches/{match_id}")
        if resp.status_code == 429:
            wait = 5 * (attempt + 1)  # back off: 5s, 10s, 15s
            print(f"  rate limited on {match_id}, waiting {wait}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        data = resp.json()
        players = data.get("players", [])
        radiant_roster = sorted([p["account_id"] for p in players if p.get("isRadiant") and p.get("account_id")])
        dire_roster = sorted([p["account_id"] for p in players if not p.get("isRadiant") and p.get("account_id")])
        return {
            "match_id": match_id,
            "radiant_roster": radiant_roster,
            "dire_roster": dire_roster,
            "is_parsed": data.get("version") is not None,
            "patch": data.get("patch"),
        }
    raise RuntimeError(f"gave up on {match_id} after {max_retries} retries")

cache_path = "data/match_detail_cache_v2.jsonl"
os.makedirs("data", exist_ok=True)

seen_ids = set()
detail_records = []
if os.path.exists(cache_path):
    with open(cache_path) as f:
        for line in f:
            rec = json.loads(line)
            detail_records.append(rec)
            seen_ids.add(rec["match_id"])
    print(f"resuming: {len(seen_ids)} already cached")

with open(cache_path, "a") as f:
    for i, match_id in enumerate(df_reliable["match_id"]):
        if match_id in seen_ids:
            continue
        try:
            rec = get_match_detail(match_id)
            detail_records.append(rec)
            f.write(json.dumps(rec) + "\n")
        except Exception as e:
            print(f"failed on {match_id}: {e}")
        time.sleep(1.5)
        if i % 50 == 0:
            print(f"{i}/{len(df_reliable)}")

df_rosters = pd.DataFrame(detail_records)
print(df_rosters.shape)

## Merge and save


In [ ]:
print(f"df_reliable: {df_reliable.shape}")
print(f"df_rosters: {df_rosters.shape}")

df_raw = df_reliable.merge(df_rosters, on="match_id", how="inner")
print(f"df_raw after merge: {df_raw.shape}")

df_raw["radiant_roster_str"] = df_raw["radiant_roster"].apply(lambda r: ";".join(map(str, r)))
df_raw["dire_roster_str"] = df_raw["dire_roster"].apply(lambda r: ";".join(map(str, r)))

cols_to_save = [
    "match_id", "start_time", "leagueid", "league_name", "series_type",
    "radiant_team_id", "radiant_name", "dire_team_id", "dire_name",
    "radiant_win", "is_parsed", "patch",
    "radiant_roster_str", "dire_roster_str",
]
df_raw[cols_to_save].to_csv("data/pro_matches_raw.csv", index=False)
print("saved data/pro_matches_raw.csv")

print(f"\npatch value counts:\n{df_raw['patch'].value_counts()}")
print(f"\nseries_type value counts:\n{df_raw['series_type'].value_counts()}")


In [ ]:
series_lengths = df_raw.groupby(['series_id', 'series_type']).size().reset_index(name='games_in_series')
print(series_lengths.groupby('series_type')['games_in_series'].value_counts())